# Quantization Methods for LLMs

## Historical Context

**Quantization** - representing weights/activations with fewer bits - has evolved dramatically with LLMs:

### Timeline

#### Early Days (2015-2019)
- **2015**: BinaryNet, XNOR-Net (1-bit weights)
  - Extreme compression but severe accuracy loss
- **2017-2018**: INT8 quantization for CNNs
  - TensorRT, TensorFlow Lite
  - Post-training quantization (PTQ)
  - ~2-4x speedup with <1% accuracy loss

#### LLM Era Begins (2020-2021)
- **2020**: ZeroQuant (Microsoft)
  - Group-wise quantization
  - Better for transformers than channel-wise
- **2021**: LLM.int8() (Dettmers et al.)
  - Mixed-precision decomposition
  - Outlier handling critical for LLMs
  - Enabled inference on smaller GPUs

#### Advanced Methods (2022-2023)
- **2022**: GPTQ (Frantar et al., October)
  - **Key innovation**: Optimal brain quantization for GPT models
  - 4-bit/3-bit with minimal quality loss
  - Fast GPU inference
  - Enables 13B models on consumer GPUs

- **2023**: Multiple breakthroughs
  - **AWQ** (Lin et al., June): Activation-aware weight quantization
    - Protect salient weights based on activation magnitude
    - Better quality than GPTQ at same bits
  - **GGUF/GGML** (Gerganov): llama.cpp format
    - Optimized for CPU inference
    - k-quant methods (mixed precision within tensor)
    - Democratized LLM deployment
  - **QuIP/QuIP#** (Chee et al.): 2-bit quantization
    - Incoherence processing
    - Push limits of extreme quantization

- **2024**: Production adoption
  - GPTQ/AWQ standard for GPU deployment
  - GGUF dominant for CPU/edge
  - INT4/FP4 in commercial hardware (Nvidia H100)

### Why Quantization Matters

For a 70B parameter LLM:
- **FP16**: 140 GB memory
- **INT8**: 70 GB (fits on A100)
- **INT4 (GPTQ/AWQ)**: 35 GB (fits on 4x3090)
- **INT4 (GGUF)**: CPU inference possible!

### Quality vs Compression Trade-off

| Method | Bits | Speedup | Quality | Use Case |
|--------|------|---------|---------|----------|
| FP16 | 16 | 1x | 100% | Training, high-end inference |
| INT8 | 8 | 2-3x | 99%+ | Standard inference |
| GPTQ/AWQ (4-bit) | 4 | 3-4x | 95-98% | Consumer GPU inference |
| GGUF Q4 | 4-6 | 4-5x | 93-97% | CPU/edge inference |
| Extreme (2-3 bit) | 2-3 | 5-8x | 85-95% | Resource-constrained |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Optional
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Check for quantization libraries
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print(f"bitsandbytes version: {bnb.__version__}")
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes not available (pip install bitsandbytes)")

try:
    from auto_gptq import AutoGPTQForCausalLM
    GPTQ_AVAILABLE = True
    print("auto-gptq available")
except ImportError:
    GPTQ_AVAILABLE = False
    print("auto-gptq not available (pip install auto-gptq)")

## Quantization Fundamentals

### Basic Concepts

**Quantization** maps continuous values to discrete set:

```
Quantization: x_q = round(x / scale) + zero_point
Dequantization: x ≈ (x_q - zero_point) * scale
```

### Types of Quantization

1. **Symmetric vs Asymmetric**
   - Symmetric: zero_point = 0, range symmetric around 0
   - Asymmetric: zero_point ≠ 0, better for asymmetric distributions

2. **Per-Tensor vs Per-Channel vs Per-Group**
   - Per-tensor: Single scale for entire tensor
   - Per-channel: Scale per output channel (better accuracy)
   - Per-group: Scale per group of elements (GPTQ/AWQ)

3. **Post-Training (PTQ) vs Quantization-Aware Training (QAT)**
   - PTQ: Quantize after training (faster, less accurate)
   - QAT: Train with quantization simulation (slower, more accurate)

In [ ]:
def quantize_tensor(
    x: torch.Tensor,
    n_bits: int = 8,
    symmetric: bool = True,
    per_channel: bool = False,
    channel_dim: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Basic quantization implementation.
    
    Args:
        x: Input tensor
        n_bits: Number of bits (4, 8, etc.)
        symmetric: Use symmetric quantization
        per_channel: Use per-channel scaling
        channel_dim: Which dimension is channel
    
    Returns:
        quantized: Quantized tensor (int)
        scale: Scaling factor(s)
        zero_point: Zero point(s) if asymmetric
    """
    # Quantization range
    if symmetric:
        q_min = -(2 ** (n_bits - 1))
        q_max = 2 ** (n_bits - 1) - 1
    else:
        q_min = 0
        q_max = 2 ** n_bits - 1
    
    # Calculate scale and zero_point
    if per_channel:
        # Compute per-channel statistics
        shape = [1] * x.ndim
        shape[channel_dim] = x.shape[channel_dim]
        
        x_min = x.amin(dim=[d for d in range(x.ndim) if d != channel_dim], keepdim=True)
        x_max = x.amax(dim=[d for d in range(x.ndim) if d != channel_dim], keepdim=True)
    else:
        x_min = x.min()
        x_max = x.max()
    
    if symmetric:
        # Symmetric: scale based on max absolute value
        abs_max = torch.max(x_min.abs(), x_max.abs())
        scale = abs_max / q_max
        zero_point = None
    else:
        # Asymmetric: scale based on range
        scale = (x_max - x_min) / (q_max - q_min)
        zero_point = q_min - (x_min / scale).round()
    
    # Avoid division by zero
    scale = torch.clamp(scale, min=1e-8)
    
    # Quantize
    if symmetric:
        x_q = (x / scale).round().clamp(q_min, q_max)
    else:
        x_q = ((x / scale).round() + zero_point).clamp(q_min, q_max)
    
    return x_q, scale, zero_point


def dequantize_tensor(
    x_q: torch.Tensor,
    scale: torch.Tensor,
    zero_point: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Dequantize tensor."""
    if zero_point is None:
        return x_q * scale
    else:
        return (x_q - zero_point) * scale


# Test basic quantization
x = torch.randn(100, 100)

# Visualize quantization for different bit widths
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

bit_widths = [8, 4, 3, 2]

for idx, n_bits in enumerate(bit_widths):
    x_q, scale, zero_point = quantize_tensor(x, n_bits=n_bits, symmetric=True)
    x_dq = dequantize_tensor(x_q, scale, zero_point)
    
    error = (x - x_dq).abs()
    
    # Original vs quantized
    ax = axes[idx]
    ax.scatter(x.flatten()[:1000], x_dq.flatten()[:1000], alpha=0.3, s=1)
    ax.plot([-4, 4], [-4, 4], 'r--', label='Perfect reconstruction')
    ax.set_xlabel('Original')
    ax.set_ylabel('Quantized')
    ax.set_title(f'{n_bits}-bit (MSE: {error.pow(2).mean():.4f})')
    ax.legend()
    ax.grid(alpha=0.3)

# Error distribution
ax = axes[4]
for n_bits in bit_widths:
    x_q, scale, zero_point = quantize_tensor(x, n_bits=n_bits, symmetric=True)
    x_dq = dequantize_tensor(x_q, scale, zero_point)
    error = (x - x_dq).abs()
    ax.hist(error.flatten().numpy(), bins=50, alpha=0.5, label=f'{n_bits}-bit')
ax.set_xlabel('Absolute Error')
ax.set_ylabel('Frequency')
ax.set_title('Error Distribution')
ax.legend()
ax.grid(alpha=0.3)

# MSE vs bits
ax = axes[5]
bits = [2, 3, 4, 5, 6, 7, 8]
mses = []
for n_bits in bits:
    x_q, scale, zero_point = quantize_tensor(x, n_bits=n_bits, symmetric=True)
    x_dq = dequantize_tensor(x_q, scale, zero_point)
    mse = (x - x_dq).pow(2).mean().item()
    mses.append(mse)
ax.semilogy(bits, mses, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Bit Width')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Quantization Error vs Bit Width')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quantization_fundamentals.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQuantization Error Analysis:")
for n_bits in [8, 4, 3, 2]:
    x_q, scale, zero_point = quantize_tensor(x, n_bits=n_bits, symmetric=True)
    x_dq = dequantize_tensor(x_q, scale, zero_point)
    mse = (x - x_dq).pow(2).mean().item()
    max_error = (x - x_dq).abs().max().item()
    print(f"  {n_bits}-bit: MSE={mse:.6f}, Max Error={max_error:.6f}")

## Post-Training Quantization (PTQ)

Simplest approach: quantize trained model without retraining.

### Calibration
Need calibration data to determine quantization parameters:
1. Run calibration samples through model
2. Collect activation statistics
3. Determine optimal scales/zero-points

### Methods
- **Min-Max**: Use min/max of activations
- **Percentile**: Clip outliers (e.g., 99.9th percentile)
- **MSE**: Minimize quantization error
- **Entropy**: Minimize KL divergence

In [ ]:
class QuantizedLinear(nn.Module):
    """
    Quantized linear layer (INT8 weights).
    
    This is a simplified version showing the concept.
    Production implementations use specialized kernels.
    """
    
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Store quantized weights (INT8)
        self.register_buffer('weight_q', torch.zeros(out_features, in_features, dtype=torch.int8))
        self.register_buffer('weight_scale', torch.zeros(out_features))
        
        if bias:
            self.register_buffer('bias', torch.zeros(out_features))
        else:
            self.bias = None
    
    @classmethod
    def from_float(cls, module: nn.Linear):
        """Convert float linear layer to quantized."""
        quant_module = cls(module.in_features, module.out_features, module.bias is not None)
        
        # Quantize weights (per-channel)
        weight_q, weight_scale, _ = quantize_tensor(
            module.weight.data,
            n_bits=8,
            symmetric=True,
            per_channel=True,
            channel_dim=0,
        )
        
        quant_module.weight_q = weight_q.to(torch.int8)
        quant_module.weight_scale = weight_scale.squeeze()
        
        if module.bias is not None:
            quant_module.bias = module.bias.data.clone()
        
        return quant_module
    
    def forward(self, x):
        # Dequantize weights for computation
        # In practice, use INT8 matmul for speedup
        weight_dq = self.weight_q.float() * self.weight_scale.unsqueeze(1)
        output = F.linear(x, weight_dq, self.bias)
        return output
    
    def extra_repr(self):
        return f'in_features={self.in_features}, out_features={self.out_features}, quantized=INT8'


# Test quantized linear
original = nn.Linear(512, 256, bias=True)
quantized = QuantizedLinear.from_float(original)

x = torch.randn(16, 128, 512)

with torch.no_grad():
    output_orig = original(x)
    output_quant = quantized(x)

error = (output_orig - output_quant).abs()

print("Quantized Linear Layer Test:")
print(f"  Original size: {original.weight.element_size() * original.weight.numel() / 1e6:.2f} MB")
print(f"  Quantized size: {quantized.weight_q.element_size() * quantized.weight_q.numel() / 1e6:.2f} MB")
print(f"  Compression: {original.weight.element_size() / quantized.weight_q.element_size():.1f}x")
print(f"\n  Max error: {error.max():.6f}")
print(f"  Mean error: {error.mean():.6f}")
print(f"  Relative error: {(error / output_orig.abs().mean()).mean():.2%}")

## GPTQ: Optimal Post-Training Quantization

**GPTQ** (Frantar et al., 2022) applies Optimal Brain Quantization to GPT-style models.

### Key Ideas

1. **Layer-wise Quantization**: Process one layer at a time
2. **Hessian-guided**: Use second-order information for optimal quantization
3. **Error Compensation**: Adjust remaining weights to compensate for quantization error

### Algorithm (Simplified)
```
For each layer:
    1. Collect activations from calibration data
    2. Compute Hessian: H = (X^T X) / n_samples
    3. For each column of weights:
        - Quantize column
        - Compute quantization error
        - Update remaining columns to compensate
    4. Continue to next layer
```

### Advantages
- 4-bit/3-bit with minimal quality loss
- Fast GPU inference (with specialized kernels)
- Per-group quantization for better accuracy

### Trade-offs
- Requires calibration data (~128-1024 samples)
- Slow quantization process (minutes to hours)
- Once quantized, weights are fixed

In [ ]:
def gptq_quantize_layer(
    weight: torch.Tensor,
    H: torch.Tensor,
    n_bits: int = 4,
    group_size: int = 128,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Simplified GPTQ quantization for one layer.
    
    Args:
        weight: [out_features, in_features]
        H: Hessian matrix [in_features, in_features]
        n_bits: Bit width
        group_size: Group size for per-group quantization
    
    Returns:
        weight_q: Quantized weight
        scales: Quantization scales per group
    """
    out_features, in_features = weight.shape
    n_groups = (in_features + group_size - 1) // group_size
    
    # Quantization range
    q_max = 2 ** (n_bits - 1) - 1
    
    weight_q = torch.zeros_like(weight)
    scales = torch.zeros(out_features, n_groups)
    
    # Process each group
    for g in range(n_groups):
        start = g * group_size
        end = min(start + group_size, in_features)
        
        # Get weights for this group
        w_group = weight[:, start:end]
        
        # Compute scale for this group
        scale = w_group.abs().max(dim=1)[0] / q_max
        scale = scale.clamp(min=1e-8)
        
        # Quantize
        w_q = (w_group / scale.unsqueeze(1)).round().clamp(-q_max, q_max)
        
        # In full GPTQ, would update remaining weights here
        # to compensate for quantization error
        
        weight_q[:, start:end] = w_q
        scales[:, g] = scale
    
    return weight_q, scales


def gptq_dequantize(
    weight_q: torch.Tensor,
    scales: torch.Tensor,
    group_size: int = 128,
) -> torch.Tensor:
    """Dequantize GPTQ weights."""
    out_features, in_features = weight_q.shape
    weight = torch.zeros_like(weight_q, dtype=torch.float32)
    
    n_groups = scales.shape[1]
    
    for g in range(n_groups):
        start = g * group_size
        end = min(start + group_size, in_features)
        
        scale = scales[:, g].unsqueeze(1)
        weight[:, start:end] = weight_q[:, start:end] * scale
    
    return weight


# Test GPTQ-style quantization
weight = torch.randn(256, 512)

# Simulate Hessian (in practice, computed from activations)
H = torch.eye(512) + torch.randn(512, 512) * 0.01
H = (H + H.T) / 2  # Make symmetric

# Quantize
weight_q, scales = gptq_quantize_layer(weight, H, n_bits=4, group_size=128)
weight_dq = gptq_dequantize(weight_q, scales, group_size=128)

error = (weight - weight_dq).abs()

print("GPTQ-style Quantization Test:")
print(f"  Bit width: 4")
print(f"  Group size: 128")
print(f"  Number of groups: {scales.shape[1]}")
print(f"\n  Original size: {weight.element_size() * weight.numel() / 1e6:.2f} MB")
print(f"  Quantized size: {(weight_q.element_size() * weight_q.numel() / 2 + scales.element_size() * scales.numel()) / 1e6:.2f} MB")
print(f"  Compression: ~4x (16-bit -> 4-bit)")
print(f"\n  Max error: {error.max():.6f}")
print(f"  Mean error: {error.mean():.6f}")
print(f"  Relative error: {(error / weight.abs().mean()).mean():.2%}")

## AWQ: Activation-Aware Weight Quantization

**AWQ** (Lin et al., 2023) improves on GPTQ by protecting important weights.

### Key Insight

Not all weights are equally important!
- Weights with large activations have more impact
- Protect salient weights (keep higher precision)
- Quantize less important weights more aggressively

### Algorithm
```
1. Collect activation magnitudes from calibration
2. Compute salience: S_i = avg(|activation_i|)
3. Scale weights before quantization:
   w_scaled = w * s^α where s is per-channel salience
4. Quantize scaled weights
5. Scale back during inference
```

### Advantages over GPTQ
- Better quality at same bit width
- Simpler algorithm (no Hessian)
- Faster quantization
- More robust to calibration data

### Trade-offs
- Requires activation collection
- Slightly more complex inference (scaling)

In [ ]:
def awq_compute_salience(
    activations: torch.Tensor,
) -> torch.Tensor:
    """
    Compute per-channel salience from activations.
    
    Args:
        activations: [batch, seq_len, in_features] or list of such tensors
    
    Returns:
        salience: [in_features] average activation magnitude
    """
    if isinstance(activations, list):
        # Multiple batches
        all_acts = torch.cat([a.reshape(-1, a.shape[-1]) for a in activations], dim=0)
    else:
        all_acts = activations.reshape(-1, activations.shape[-1])
    
    # Average absolute activation per channel
    salience = all_acts.abs().mean(dim=0)
    
    return salience


def awq_quantize_layer(
    weight: torch.Tensor,
    salience: torch.Tensor,
    n_bits: int = 4,
    group_size: int = 128,
    alpha: float = 0.5,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    AWQ quantization.
    
    Args:
        weight: [out_features, in_features]
        salience: [in_features] activation magnitudes
        n_bits: Bit width
        group_size: Group size
        alpha: Scaling exponent (0.5 works well)
    
    Returns:
        weight_q: Quantized weights
        scales: Quantization scales
        salience_scales: Per-channel salience scales
    """
    out_features, in_features = weight.shape
    
    # Compute per-channel scaling factors
    # Protect salient channels by scaling them up before quantization
    salience_normalized = salience / salience.mean()
    salience_scales = salience_normalized.pow(alpha)
    
    # Scale weights
    weight_scaled = weight * salience_scales.unsqueeze(0)
    
    # Quantize scaled weights (using GPTQ-style group quantization)
    weight_q, scales = gptq_quantize_layer(
        weight_scaled,
        H=None,  # Not needed for AWQ
        n_bits=n_bits,
        group_size=group_size,
    )
    
    return weight_q, scales, salience_scales


def awq_dequantize(
    weight_q: torch.Tensor,
    scales: torch.Tensor,
    salience_scales: torch.Tensor,
    group_size: int = 128,
) -> torch.Tensor:
    """Dequantize AWQ weights."""
    # Dequantize
    weight = gptq_dequantize(weight_q, scales, group_size)
    
    # Scale back
    weight = weight / salience_scales.unsqueeze(0)
    
    return weight


# Test AWQ quantization
weight = torch.randn(256, 512)

# Simulate activations (in practice, collected from calibration data)
# Some channels more active than others
activations = torch.randn(16, 128, 512)
activations[:, :, :100] *= 3.0  # Make first 100 channels more salient

salience = awq_compute_salience(activations)

# Compare GPTQ vs AWQ
print("\nGPTQ vs AWQ Comparison:\n")

# GPTQ
weight_q_gptq, scales_gptq = gptq_quantize_layer(weight, H=None, n_bits=4, group_size=128)
weight_dq_gptq = gptq_dequantize(weight_q_gptq, scales_gptq, group_size=128)
error_gptq = (weight - weight_dq_gptq).abs()

print("GPTQ (4-bit):")
print(f"  Max error: {error_gptq.max():.6f}")
print(f"  Mean error: {error_gptq.mean():.6f}")
print(f"  Relative error: {(error_gptq / weight.abs().mean()).mean():.2%}")

# AWQ
weight_q_awq, scales_awq, salience_scales = awq_quantize_layer(
    weight, salience, n_bits=4, group_size=128, alpha=0.5
)
weight_dq_awq = awq_dequantize(weight_q_awq, scales_awq, salience_scales, group_size=128)
error_awq = (weight - weight_dq_awq).abs()

print("\nAWQ (4-bit):")
print(f"  Max error: {error_awq.max():.6f}")
print(f"  Mean error: {error_awq.mean():.6f}")
print(f"  Relative error: {(error_awq / weight.abs().mean()).mean():.2%}")

print(f"\nAWQ improvement: {(error_gptq.mean() / error_awq.mean()):.2f}x lower error")

# Visualize salience effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Salience distribution
axes[0].hist(salience.numpy(), bins=50, alpha=0.7)
axes[0].set_xlabel('Salience (Activation Magnitude)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Channel Salience Distribution')
axes[0].axvline(salience[:100].mean(), color='r', linestyle='--', label='High salience channels')
axes[0].axvline(salience[100:].mean(), color='b', linestyle='--', label='Low salience channels')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Error comparison per channel
axes[1].plot(error_gptq.mean(dim=0).numpy(), alpha=0.7, label='GPTQ')
axes[1].plot(error_awq.mean(dim=0).numpy(), alpha=0.7, label='AWQ')
axes[1].axvline(100, color='k', linestyle='--', alpha=0.5, label='Salience boundary')
axes[1].set_xlabel('Channel Index')
axes[1].set_ylabel('Mean Quantization Error')
axes[1].set_title('Per-Channel Quantization Error')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Salience scaling effect
axes[2].plot(salience_scales.numpy(), alpha=0.7)
axes[2].axhline(1.0, color='r', linestyle='--', alpha=0.5, label='No scaling')
axes[2].set_xlabel('Channel Index')
axes[2].set_ylabel('Salience Scale Factor')
axes[2].set_title('AWQ Scaling Factors (α=0.5)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('awq_vs_gptq.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insight: AWQ protects high-salience channels, reducing error where it matters most.")

## GGUF Format for llama.cpp

**GGUF** (GPT-Generated Unified Format) is optimized for CPU inference.

### Key Features

1. **Mixed Precision**: Different bit widths for different tensors
2. **K-Quant**: Mixed precision within a single tensor
3. **CPU-Optimized**: SIMD instructions, cache-friendly layout
4. **Flexible**: Q2, Q3, Q4, Q5, Q6, Q8 variants

### GGUF Quantization Types

| Type | Bits | Description |
|------|------|-------------|
| Q2_K | ~2.6 | Extremely compressed, quality loss |
| Q3_K_S | ~3.5 | Small, good for 7B models |
| Q4_K_M | ~4.5 | Most popular, best quality/size |
| Q5_K_M | ~5.5 | High quality, larger |
| Q6_K | ~6.6 | Very high quality |
| Q8_0 | 8 | Minimal quality loss |

### K-Quant Strategy

Divide weights into blocks:
- Important blocks: Higher precision
- Less important: Lower precision
- Importance based on magnitude distribution

### Use Cases
- **Edge devices**: Run 7B models on phones
- **CPU inference**: No GPU required
- **Low-end GPUs**: Reduce VRAM requirements
- **Local deployment**: Privacy-focused applications

In [ ]:
def gguf_k_quant_block(
    weights: torch.Tensor,
    block_size: int = 256,
    n_bits_high: int = 6,
    n_bits_low: int = 4,
    high_importance_ratio: float = 0.25,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Simplified K-quant implementation.
    
    Args:
        weights: Weight tensor
        block_size: Size of each block
        n_bits_high: Bits for important blocks
        n_bits_low: Bits for less important blocks
        high_importance_ratio: Fraction of blocks to use high precision
    
    Returns:
        quantized: Quantized weights
        scales: Per-block scales
        is_high_importance: Boolean mask for high-importance blocks
    """
    flat_weights = weights.flatten()
    n_elements = flat_weights.numel()
    n_blocks = (n_elements + block_size - 1) // block_size
    
    # Pad to multiple of block_size
    padded_size = n_blocks * block_size
    if padded_size > n_elements:
        flat_weights = F.pad(flat_weights, (0, padded_size - n_elements))
    
    blocks = flat_weights.reshape(n_blocks, block_size)
    
    # Compute importance of each block (based on max magnitude)
    block_importance = blocks.abs().max(dim=1)[0]
    
    # Determine which blocks get high precision
    n_high_blocks = int(n_blocks * high_importance_ratio)
    importance_threshold = torch.topk(block_importance, n_high_blocks, sorted=False)[0].min()
    is_high_importance = block_importance >= importance_threshold
    
    # Quantize each block
    quantized = torch.zeros_like(blocks)
    scales = torch.zeros(n_blocks)
    
    for i in range(n_blocks):
        n_bits = n_bits_high if is_high_importance[i] else n_bits_low
        q_max = 2 ** (n_bits - 1) - 1
        
        block = blocks[i]
        scale = block.abs().max() / q_max
        scale = scale.clamp(min=1e-8)
        
        quantized[i] = (block / scale).round().clamp(-q_max, q_max)
        scales[i] = scale
    
    # Reshape back
    quantized = quantized.flatten()[:n_elements].reshape(weights.shape)
    
    return quantized, scales, is_high_importance


# Test K-quant
weight = torch.randn(256, 512)

# Uniform quantization (4-bit)
weight_q_uniform, scale_uniform, _ = quantize_tensor(weight, n_bits=4, symmetric=True)
weight_dq_uniform = dequantize_tensor(weight_q_uniform, scale_uniform)
error_uniform = (weight - weight_dq_uniform).abs().mean()

# K-quant (mixed 4-bit and 6-bit)
weight_q_kquant, scales_kquant, is_high = gguf_k_quant_block(
    weight, block_size=256, n_bits_high=6, n_bits_low=4, high_importance_ratio=0.25
)

# Dequantize K-quant
flat_q = weight_q_kquant.flatten()
n_blocks = scales_kquant.numel()
block_size = 256
flat_dq = torch.zeros_like(flat_q, dtype=torch.float32)
for i in range(n_blocks):
    start = i * block_size
    end = min(start + block_size, flat_q.numel())
    flat_dq[start:end] = flat_q[start:end] * scales_kquant[i]
weight_dq_kquant = flat_dq.reshape(weight.shape)
error_kquant = (weight - weight_dq_kquant).abs().mean()

print("\nGGUF K-Quant Test:\n")
print(f"Uniform 4-bit:")
print(f"  Mean error: {error_uniform:.6f}")
print(f"  Average bits: 4.0")

print(f"\nK-Quant (25% at 6-bit, 75% at 4-bit):")
print(f"  Mean error: {error_kquant:.6f}")
print(f"  Average bits: {0.25 * 6 + 0.75 * 4:.1f}")
print(f"  Quality improvement: {(error_uniform / error_kquant - 1) * 100:.1f}%")
print(f"  High-importance blocks: {is_high.sum().item()} / {len(is_high)}")

# Compare sizes for different GGUF formats
print("\n\nGGUF Format Comparison (for 7B model):")
print(f"{'Format':<15} {'Size (GB)':<12} {'Relative Quality':<20} {'Use Case':<30}")
print("-" * 80)

base_size = 7 * 2  # 7B params * 2 bytes (FP16)
formats = [
    ('FP16', 16, base_size * 16 / 16, '100%', 'Training, reference'),
    ('Q8_0', 8, base_size * 8 / 16, '99.9%', 'High quality, GPU'),
    ('Q6_K', 6.6, base_size * 6.6 / 16, '99.5%', 'Very high quality'),
    ('Q5_K_M', 5.5, base_size * 5.5 / 16, '99%', 'High quality, balanced'),
    ('Q4_K_M', 4.5, base_size * 4.5 / 16, '97-98%', 'Most popular'),
    ('Q3_K_M', 3.5, base_size * 3.5 / 16, '95-96%', 'Good compression'),
    ('Q2_K', 2.6, base_size * 2.6 / 16, '90-93%', 'Maximum compression'),
]

for fmt, bits, size_gb, quality, use_case in formats:
    print(f"{fmt:<15} {size_gb:<12.1f} {quality:<20} {use_case:<30}")

print("\nNote: Quality percentages are approximate and depend on model architecture.")

## Choosing a Quantization Method

### Decision Matrix

#### For GPU Inference
- **Have calibration data + GPU**: **AWQ** (best quality/performance)
- **No calibration data**: **bitsandbytes NF4** (zero-shot quantization)
- **Need maximum speed**: **GPTQ** with CUDA kernels
- **Conservative approach**: **INT8** via PyTorch/TensorRT

#### For CPU Inference
- **Always use GGUF format** (llama.cpp)
- **Model < 7B**: Q4_K_M or Q5_K_M
- **Model 7B-13B**: Q4_K_M
- **Model 13B-70B**: Q3_K_M or Q4_K_M
- **Extreme compression**: Q2_K (expect quality loss)

#### Quality Requirements
- **No quality loss**: INT8 (LLM.int8(), Q8_0)
- **<2% loss**: AWQ 4-bit, GPTQ 4-bit, Q5_K_M
- **2-5% loss**: Q4_K_M, Q3_K_M
- **5-10% loss**: Q2_K

### Practical Examples

**Example 1: Deploy LLaMA-2 70B**
- GPU (A100 80GB): AWQ 4-bit (~35GB)
- GPU (4x 3090 24GB): GPTQ 4-bit + tensor parallelism
- CPU (128GB RAM): GGUF Q4_K_M (~39GB)

**Example 2: Edge deployment (Mistral 7B)**
- M1 MacBook: GGUF Q4_K_M (~4GB)
- Android phone: GGUF Q2_K (~2GB)
- Low-end GPU: GPTQ 4-bit

**Example 3: Production API**
- High throughput: INT8 with TensorRT
- Cost-optimized: AWQ 4-bit on T4/A10G
- Quality-critical: FP16 on A100

In [ ]:
# Visualize quantization method comparison
methods = [
    ('FP16', 16, 100.0, 1.0, 140.0),
    ('INT8', 8, 99.5, 2.5, 70.0),
    ('AWQ 4-bit', 4, 97.5, 3.5, 35.0),
    ('GPTQ 4-bit', 4, 96.5, 3.8, 35.0),
    ('GGUF Q4_K_M', 4.5, 97.0, 4.0, 39.2),
    ('GGUF Q3_K_M', 3.5, 95.5, 5.0, 30.6),
    ('GGUF Q2_K', 2.6, 91.5, 6.0, 22.7),
]

labels = [m[0] for m in methods]
bits = [m[1] for m in methods]
quality = [m[2] for m in methods]
speedup = [m[3] for m in methods]
memory = [m[4] for m in methods]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Quality vs Memory
ax = axes[0, 0]
scatter = ax.scatter(memory, quality, s=[b*30 for b in bits], alpha=0.6, c=speedup, cmap='viridis')
for i, label in enumerate(labels):
    ax.annotate(label, (memory[i], quality[i]), fontsize=9, ha='right')
ax.set_xlabel('Memory (GB, for 70B model)')
ax.set_ylabel('Relative Quality (%)')
ax.set_title('Quality vs Memory Trade-off')
ax.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Speedup')

# Bar chart: Memory
ax = axes[0, 1]
ax.barh(range(len(labels)), memory, alpha=0.7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel('Memory (GB)')
ax.set_title('Memory Requirements (70B Model)')
ax.grid(axis='x', alpha=0.3)

# Bar chart: Quality
ax = axes[1, 0]
colors = ['green' if q >= 97 else 'orange' if q >= 95 else 'red' for q in quality]
ax.barh(range(len(labels)), quality, alpha=0.7, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel('Relative Quality (%)')
ax.set_title('Model Quality')
ax.axvline(97, color='g', linestyle='--', alpha=0.5, label='High quality')
ax.axvline(95, color='orange', linestyle='--', alpha=0.5, label='Acceptable')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Speedup
ax = axes[1, 1]
ax.bar(range(len(labels)), speedup, alpha=0.7)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Speedup vs FP16')
ax.set_title('Inference Speedup')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('quantization_method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQuantization Method Recommendation Guide:")
print("\n1. Priority: Maximum Quality")
print("   → INT8 or AWQ 4-bit")
print("\n2. Priority: Minimum Memory")
print("   → GGUF Q2_K or Q3_K_M")
print("\n3. Priority: Balanced")
print("   → GPTQ/AWQ 4-bit (GPU) or GGUF Q4_K_M (CPU)")
print("\n4. Priority: Speed")
print("   → GPTQ 4-bit with optimized kernels")
print("\n5. Priority: Ease of use")
print("   → GGUF (llama.cpp) - works everywhere")

## Summary

### Key Insights

1. **Quantization is Essential**
   - Makes large models accessible
   - 4x memory reduction with <3% quality loss
   - Critical for deployment

2. **Method Matters**
   - GPTQ: Fast GPU inference, good quality
   - AWQ: Best quality, activation-aware
   - GGUF: CPU/edge deployment, flexible

3. **Trade-offs**
   - Quality vs compression
   - Speed vs memory
   - GPU vs CPU optimization

4. **Calibration Matters**
   - 128-1024 samples sufficient
   - Representative data important
   - Can reuse across similar tasks

### Best Practices

1. **Start with INT8**: Minimal quality loss
2. **Evaluate carefully**: Test on your tasks
3. **Consider hardware**: GPU vs CPU optimization
4. **Use existing tools**: Don't reimplement
5. **Monitor quality**: Track metrics across versions

### Tool Recommendations

- **GPU inference**: transformers + bitsandbytes, auto-gptq, or awq
- **CPU inference**: llama.cpp with GGUF
- **Production**: TensorRT for maximum performance
- **Research**: Implement custom methods in PyTorch

### Future Directions

- **Lower bit widths**: 2-bit, 1-bit quantization
- **Hardware support**: Native INT4/FP4 (H100+)
- **Adaptive methods**: Dynamic bit allocation
- **Training quantization**: QLoRA, BitFit

### Resources

- [GPTQ Paper](https://arxiv.org/abs/2210.17323)
- [AWQ Paper](https://arxiv.org/abs/2306.00978)
- [LLM.int8() Paper](https://arxiv.org/abs/2208.07339)
- [llama.cpp](https://github.com/ggerganov/llama.cpp)
- [bitsandbytes](https://github.com/TimDettmers/bitsandbytes)
- [auto-gptq](https://github.com/PanQiWei/AutoGPTQ)
- [AutoAWQ](https://github.com/casper-hansen/AutoAWQ)